In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Общая информация по данным

В данной работе используется DDoS Dataset. 

Данный набор данных предназначен для разработки методов машинного обучения для обнаружения атак типа DDoS в сетевом трафике.

Источник данных: https://www.kaggle.com/datasets/devendra416/ddos-datasets

Датасет был сформирован на основе нескольких публичных наборов данных для систем обнаружения вторжений (IDS): 

CSE-CIC-IDS2018-AWS, CICIDS2017, CIC DoS dataset(2016)

Данный датасет может использоваться для решения задач:

- обнаружения сетевых атак;

- классификации сетевого трафика;

- разработки систем обнаружения вторжений (IDS);

- анализа поведения сетевых потоков.

Методы машинного обучения позволяют на основе признаков сетевого потока автоматически определять, является ли данный поток нормальным трафиком или частью DDoS атаки.

# Описание целевой задачи

В данном наборе данных представлены характеристики сетевых потоков, содержащих как обычный сетевой трафик, так и трафик, связанный с DDoS-атаками.

Исходя из структуры данных, целевой задачей анализа является классификация сетевого трафика. Необходимо на основе признаков сетевого потока определить, относится ли данный поток к нормальному сетевому трафику или к трафику, связанному с DDoS-атакой.

In [45]:
df = pd.read_csv('unbalaced_20_80_dataset.csv')
df.head(5)

,Unnamed: 0,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,1739476,172.31.69.25-18.219.193.20-80-37882-6,18.219.193.20,37882,172.31.69.25,80,6,16/02/2018 11:27:29 PM,8660,1,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ddos
1,1822666,172.31.69.28-18.219.9.1-80-63287-6,172.31.69.28,80,18.219.9.1,63287,6,22/02/2018 12:13:52 AM,5829,4,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ddos
2,905739,172.31.69.28-52.14.136.135-80-63095-6,52.14.136.135,63095,172.31.69.28,80,6,22/02/2018 12:14:02 AM,3396,1,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ddos
3,1143064,172.31.69.28-18.216.200.189-80-52341-6,18.216.200.189,52341,172.31.69.28,80,6,22/02/2018 12:28:04 AM,2390,1,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ddos
4,1934016,172.31.69.28-18.218.55.126-80-57459-6,172.31.69.28,80,18.218.55.126,57459,6,22/02/2018 12:19:45 AM,17362,4,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ddos


In [24]:
print("Размер датасета:", df.shape)
print("Типы признаков:")
print(df.dtypes)

print("Распределение классов:")
print(df['Label'].value_counts())

Размер датасета: (7616509, 85)
Типы признаков:
Unnamed: 0      int64
Flow ID        object
Src IP         object
Src Port        int64
Dst IP         object
               ...   
Idle Mean     float64
Idle Std      float64
Idle Max      float64
Idle Min      float64
Label          object
Length: 85, dtype: object
Распределение классов:
Label
Benign    6321980
ddos      1294529
Name: count, dtype: int64


In [25]:
pd.set_option('display.max_rows', None)
print(df.dtypes)

Unnamed: 0             int64
Flow ID               object
Src IP                object
Src Port               int64
Dst IP                object
Dst Port               int64
Protocol               int64
Timestamp             object
Flow Duration          int64
Tot Fwd Pkts           int64
Tot Bwd Pkts           int64
TotLen Fwd Pkts      float64
TotLen Bwd Pkts      float64
Fwd Pkt Len Max      float64
Fwd Pkt Len Min      float64
Fwd Pkt Len Mean     float64
Fwd Pkt Len Std      float64
Bwd Pkt Len Max      float64
Bwd Pkt Len Min      float64
Bwd Pkt Len Mean     float64
Bwd Pkt Len Std      float64
Flow Byts/s          float64
Flow Pkts/s          float64
Flow IAT Mean        float64
Flow IAT Std         float64
Flow IAT Max         float64
Flow IAT Min         float64
Fwd IAT Tot          float64
Fwd IAT Mean         float64
Fwd IAT Std          float64
Fwd IAT Max          float64
Fwd IAT Min          float64
Bwd IAT Tot          float64
Bwd IAT Mean         float64
Bwd IAT Std   

# Метаинформация

В рамках лабораторной работы используется несбалансированный набор данных
unbalanced_20_80_dataset.csv, содержащий:

7 616 509 записей

85 признаков

Распределнние классов 20% DDoS-атак и 80% benign (обычный трафик)

Значения признаков в наборе данных представлены в виде числовых и категориальных данных. Большинство признаков являются числовыми (целыми и вещественными), а отдельные признаки имеют строковый формат (например, IP-адреса и временные метки). Целевой признак является категориальным.

## Описание признаков набора данных

Признаки набора данных описывают характеристики сетевых потоков (network flow).

Можно выделить следующие группы признаков:

Идентификационные признаки - содержат информацию о соединении (Flow ID, IP-адреса, временные метки) и используются только для описания, но не участвуют в обучении модели;

Параметры соединения - включают порты, протокол и длительность соединения (Src Port, Dst Port, Protocol, Flow Duration);

Характеристики пакетов - описывают количество, размер и распределение пакетов (Tot Fwd Pkts, TotLen Fwd Pkts, Pkt Len Mean и др.);

Скорости передачи данных - отражают интенсивность трафика (Flow Byts/s, Flow Pkts/s, Fwd/Bwd Pkts/s);

Временные характеристики (IAT) - интервалы между пакетами, позволяющие анализировать динамику трафика (Flow IAT, Fwd IAT, Bwd IAT);

TCP-флаги - характеризуют состояние соединения (SYN, ACK, FIN и др.);

Дополнительные характеристики потока - включают параметры сегментов, подпотоков и активных/пассивных периодов (Subflow, Active, Idle);

Целевой признак - Label, определяющий класс трафика (нормальный или DDoS-атака).

# Ограничения данных

Найдем пропущенные значения

In [26]:
pd.reset_option('display.max_rows')
df.isnull().sum()

Unnamed: 0    0
Flow ID       0
Src IP        0
Src Port      0
Dst IP        0
             ..
Idle Mean     0
Idle Std      0
Idle Max      0
Idle Min      0
Label         0
Length: 85, dtype: int64

In [27]:
df.isnull().sum().sort_values(ascending=False)

Flow Byts/s    29688
Flow ID            0
Src IP             0
Src Port           0
Unnamed: 0         0
               ...  
Idle Mean          0
Idle Std           0
Idle Max           0
Idle Min           0
Label              0
Length: 85, dtype: int64

In [28]:
df = df.dropna()

In [29]:
df.isnull().sum()

Unnamed: 0    0
Flow ID       0
Src IP        0
Src Port      0
Dst IP        0
             ..
Idle Mean     0
Idle Std      0
Idle Max      0
Idle Min      0
Label         0
Length: 85, dtype: int64

In [30]:
df = df.drop(columns=[
    'Unnamed: 0',
    'Flow ID',
    'Src IP',
    'Dst IP',
    'Timestamp'
])

In [31]:
df['Label'] = df['Label'].map({
    'Benign': 0,
    'ddos': 1
})

In [32]:
df['Label'].value_counts()

Label
0    6292297
1    1294524
Name: count, dtype: int64

In [12]:
df.describe()

,Src Port,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
count,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,...,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06,7.586821e+06
mean,4.187648e+04,1.014564e+04,9.078994e+00,1.173294e+07,1.318485e+01,6.632074e+00,6.329657e+02,5.251213e+03,2.204784e+02,1.191080e+01,...,1.239946e+01,2.229720e+05,1.142998e+05,3.350223e+05,1.461083e+05,4.146693e+06,1.669314e+05,4.285304e+06,3.991484e+06,1.706280e-01
std,2.316862e+04,2.000359e+04,5.093341e+00,3.071381e+07,1.011374e+03,3.245837e+02,3.160038e+04,6.985428e+05,3.276551e+02,2.414696e+01,...,8.509344e+00,2.976721e+06,1.813593e+06,3.944567e+06,2.503289e+06,1.421649e+07,1.755038e+06,1.452316e+07,1.405459e+07,3.761836e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.532200e+04,5.300000e+01,6.000000e+00,5.970000e+02,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,5.171000e+04,4.430000e+02,6.000000e+00,3.012000e+04,2.000000e+00,1.000000e+00,4.500000e+01,1.150000e+02,4.200000e+01,0.000000e+00,...,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,5.704800e+04,3.389000e+03,1.700000e+01,3.107215e+06,5.000000e+00,5.000000e+00,6.350000e+02,4.880000e+02,3.860000e+02,3.100000e+01,...,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,6.553500e+04,1.700000e+01,1.200000e+08,2.594430e+05,2.919230e+05,8.302176e+06,6.554527e+08,2.336000e+04,1.472000e+03,...,4.800000e+01,1.132691e+08,7.523241e+07,1.132691e+08,1.132691e+08,1.199997e+08,7.639395e+07,1.199997e+08,1.199997e+08,1.000000e+00


In [33]:
pd.set_option('display.max_rows', None)

In [34]:
np.isinf(df).sum()

Src Port                 0
Dst Port                 0
Protocol                 0
Flow Duration            0
Tot Fwd Pkts             0
Tot Bwd Pkts             0
TotLen Fwd Pkts          0
TotLen Bwd Pkts          0
Fwd Pkt Len Max          0
Fwd Pkt Len Min          0
Fwd Pkt Len Mean         0
Fwd Pkt Len Std          0
Bwd Pkt Len Max          0
Bwd Pkt Len Min          0
Bwd Pkt Len Mean         0
Bwd Pkt Len Std          0
Flow Byts/s          18067
Flow Pkts/s          18067
Flow IAT Mean            0
Flow IAT Std             0
Flow IAT Max             0
Flow IAT Min             0
Fwd IAT Tot              0
Fwd IAT Mean             0
Fwd IAT Std              0
Fwd IAT Max              0
Fwd IAT Min              0
Bwd IAT Tot              0
Bwd IAT Mean             0
Bwd IAT Std              0
Bwd IAT Max              0
Bwd IAT Min              0
Fwd PSH Flags            0
Bwd PSH Flags            0
Fwd URG Flags            0
Bwd URG Flags            0
Fwd Header Len           0
B

In [35]:
df.describe().loc[['min','max']]

,Src Port,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
min,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0
max,65535.0,65535.0,17.0,120000000.0,259443.0,291923.0,8302176.0,655452688.0,23360.0,1472.0,...,48.0,113269143.0,7.523241e+07,113269143.0,113269143.0,119999735.0,7.639395e+07,119999735.0,119999735.0,1.0


In [36]:
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

In [47]:
print("Размер датасета:", df.shape)

Размер датасета: (7568754, 80)


Ограничения данных

В ходе анализа набора данных были выявлены некоторые ограничения.

В датасете присутствуют пропущенные значения в отдельных признаках (например, в признаке Flow Byts/s). Для упрощения анализа строки с пропущенными значениями были удалены.

Также набор данных является несбалансированным: около 80% записей относятся к нормальному трафику и около 20% - к DDoS-атакам.

Для выявления возможных аномалий был проведён анализ статистических характеристик признаков (describe()).

В ходе анализа данных было обнаружено наличие бесконечных значений (inf) в признаках Flow Byts/s и Flow Pkts/s. Такие значения могут возникать при вычислении характеристик сетевых потоков, например при делении на ноль.

Для корректной работы алгоритмов машинного обучения бесконечные значения были заменены на пропущенные (NaN), после чего соответствующие строки были удалены из набора данных.

# Предлагаемые ML алгоритмы для решения целевой задачи

Для решения задачи классификации сетевого трафика были выбраны алгоритмы Random Forest и XGBoost.

Алгоритм Random Forest представляет собой ансамбль деревьев решений, который формируется путём построения множества случайных деревьев и объединения их результатов. Данный метод хорошо работает с большим количеством признаков и устойчив к шуму и выбросам в данных.

Алгоритм XGBoost относится к методам градиентного бустинга и является одним из наиболее эффективных алгоритмов машинного обучения для задач классификации. Он позволяет строить высокоточные модели за счёт последовательного улучшения предыдущих моделей.

Оба алгоритма широко применяются в задачах анализа сетевого трафика и обнаружения атак, что делает их подходящими для решения поставленной задачи классификации DDoS-трафика.

# Необходимые настройки данных для каждого алгоритма

In [38]:
df['Label'].value_counts(normalize=True)

Label
0    0.828965
1    0.171035
Name: proportion, dtype: float64

In [48]:
df_sample, _ = train_test_split(
    df,
    train_size=200000,
    stratify=df['Label'],
    random_state=42
)

In [49]:
df_sample['Label'].value_counts(normalize=True)

Label
0    0.828965
1    0.171035
Name: proportion, dtype: float64

In [50]:
df = df_sample

In [51]:
X = df.drop('Label', axis=1)
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

Размер обучающей выборки: (160000, 79)
Размер тестовой выборки: (40000, 79)


# Ожидаемые модели знаний, построенные алгоритмами

В результате применения алгоритмов Random Forest и XGBoost планируется построение моделей машинного обучения, способных выявлять закономерности между характеристиками сетевых потоков и типом трафика.

Модель Random Forest формирует ансамбль деревьев решений, каждое из которых анализирует различные подмножества признаков и данных. В результате объединения решений нескольких деревьев формируется итоговый классификатор, способный определять, относится ли сетевой поток к нормальному трафику или к DDoS-атаке.

Алгоритм XGBoost строит модель на основе метода градиентного бустинга, последовательно создавая деревья решений, которые корректируют ошибки предыдущих моделей. Это позволяет выявлять более сложные зависимости между признаками и повышает точность классификации.

Ожидается, что построенные модели смогут автоматически классифицировать сетевые потоки по их характеристикам и выявлять признаки, наиболее значимые для обнаружения DDoS-атак.

# Предлагаемые методы и критерии оценки построенных моделей

Для оценки качества построенных моделей классификации используются стандартные метрики машинного обучения. Поскольку задача заключается в классификации сетевого трафика на два класса (нормальный трафик и DDoS-атака), применяются следующие показатели качества модели.

Accuracy (точность) - показывает долю правильно классифицированных объектов среди всех объектов набора данных.

Precision (точность положительных предсказаний) - показывает долю корректно определённых атак среди всех объектов, которые модель классифицировала как атаки.

Recall (полнота) - показывает долю обнаруженных атак среди всех реальных атак в наборе данных. Данная метрика особенно важна в задачах обнаружения сетевых атак.

F1-score - гармоническое среднее между precision и recall, позволяющее учитывать баланс между этими показателями.

Также для анализа результатов классификации используется матрица ошибок (Confusion Matrix), которая позволяет наглядно оценить количество правильно и неправильно классифицированных объектов каждого класса.

Использование нескольких метрик позволяет более полно оценить эффективность построенных моделей и сравнить результаты работы алгоритмов Random Forest и XGBoost.